# Origens compartilhadas e piloto de XGBoost

Este notebook cria agendas provisórias de origens, valida que os quatro consumidores recebem a mesma agenda e executa um único ajuste de XGBoost por base. Os resultados são testes de infraestrutura, não MAE final.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'processed').exists(), 'Execute a partir da raiz do repositório.'
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd

from discovery_series_temporais.series import (
    MODEL_NAMES,
    assert_same_origins,
    build_origin_schedule,
    load_candidate_target,
    write_target_only_origin_artifacts,
)
from discovery_series_temporais.xgboost_model import run_xgboost_pilot

print(f'Raiz do projeto: {ROOT}')

Raiz do projeto: C:\Users\mayumishimizu-ieg\.codex\.chatgpt-projects\g-p-6ad36ceb09bc81918ea40186a7712f30\DiscoverySeriesTemporais


In [2]:
output_dir = ROOT / 'reports' / 'generated' / 'origin_schedules'
summary = write_target_only_origin_artifacts(ROOT / 'data' / 'processed', output_dir)

comparability_checks = []
schedules = {}
for base_id in range(1, 6):
    candidate = load_candidate_target(base_id, ROOT / 'data' / 'processed')
    schedule = build_origin_schedule(candidate)
    schedules[base_id] = schedule
    check = assert_same_origins({model: schedule.copy() for model in MODEL_NAMES}, schedule)
    comparability_checks.append({'base': base_id, **check})

display(summary)
display(pd.DataFrame(comparability_checks))

,base_id,target,frequency_interpretation,origin_count,first_origin,last_origin,origin_signature,scope
0,1,Close,daily,567,2023-03-19T00:00:00,2024-10-05T00:00:00,84c5d45450f7a8e28e1a170127f2b069327781c1525881...,target-only provisional schedule; regenerate a...
1,2,traffic_volume,hourly (target-only consolidation),7598,2017-11-16T20:00:00,2018-09-30T22:00:00,702bc19c67deb11766995748b00647c1d667d9002838e0...,target-only provisional schedule; regenerate a...
2,3,PM2.5,hourly,6786,2016-05-15T21:00:00,2017-02-28T22:00:00,16dec54399b9a0924c23e95f22d82fad9b2cd6925e8ddb...,target-only provisional schedule; regenerate a...
3,4,T (degC),hourly mean (exploratory),14008,2015-05-25T06:00:00,2016-12-31T23:00:00,00a6e41d6bf8e1bb09423e3c72af4c7b5c979a6688dd90...,target-only provisional schedule; regenerate a...
4,5,GOLD_PRICE,next trading observation,1973,2005-01-28T00:00:00,2014-04-08T00:00:00,c222b5d23dc4fa298f1d02c00d35ac046f3be8163ea1e5...,target-only provisional schedule; regenerate a...


,base,origin_count,origin_signature
0,1,567,84c5d45450f7a8e28e1a170127f2b069327781c1525881...
1,2,7598,702bc19c67deb11766995748b00647c1d667d9002838e0...
2,3,6786,16dec54399b9a0924c23e95f22d82fad9b2cd6925e8ddb...
3,4,14008,00a6e41d6bf8e1bb09423e3c72af4c7b5c979a6688dd90...
4,5,1973,c222b5d23dc4fa298f1d02c00d35ac046f3be8163ea1e5...


In [3]:
pilot_results = []
for base_id, schedule in schedules.items():
    candidate = load_candidate_target(base_id, ROOT / 'data' / 'processed')
    pilot_results.append(run_xgboost_pilot(candidate, schedule))

pilot_table = pd.DataFrame(pilot_results)
display(pilot_table[['base_id', 'origin_timestamp', 'target_timestamp', 'training_rows', 'prediction', 'actual', 'absolute_error']])

pilot_output = ROOT / 'reports' / 'generated' / 'xgboost_pilot.json'
pilot_output.write_text(json.dumps(pilot_results, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Piloto salvo em: {pilot_output.relative_to(ROOT)}')
print('Não calcular MAE ou ranking a partir desta tabela: cada linha é apenas um teste de execução.')

,base_id,origin_timestamp,target_timestamp,training_rows,prediction,actual,absolute_error
0,1,2023-03-19T00:00:00,2023-03-20T00:00:00,2261,27882.878906,27767.236328,115.642578
1,2,2017-11-16T20:00:00,2017-11-16T21:00:00,30381,2837.159912,3069.000000,231.840088
2,3,2016-05-15T21:00:00,2016-05-15T22:00:00,26561,33.694317,41.000000,7.305683
3,4,2015-05-25T06:00:00,2015-05-25T07:00:00,56016,12.461547,12.605000,0.143453
4,5,2005-01-28T00:00:00,2005-02-01T00:00:00,7883,416.789307,421.150000,4.360693


Piloto salvo em: reports\generated\xgboost_pilot.json
Não calcular MAE ou ranking a partir desta tabela: cada linha é apenas um teste de execução.
